In [8]:
import numpy as np
import time


def jacobi(A, b, x0, tol=1e-6, max_iterations=10000, blowup=1e12):
    """
    Jacobi uses only the old x values each iteration
    It will Return (x, iterations, converged_bool)
    """
    n = len(b)
    x = x0.copy().astype(float)

    D = np.diag(A)
    R = A - np.diagflat(D)  # everything except the diagonal

    for k in range(max_iterations):
        x_new = (b - R @ x) / D

        # convergence check ( infinity norm) (looked up)
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new, (k + 1), True

        # if it starts exploding or becomes inf/nan, bail
        if (not np.all(np.isfinite(x_new))) or (np.linalg.norm(x_new, ord=np.inf) > blowup):
            return x_new, (k + 1), False

        x = x_new

    return x, max_iterations, False


def gauss_seidel(A, b, x0, tol=1e-6, max_iterations=10000, blowup=1e12):
    """
    Gauss-Seidel uses updated values immediately 
    it will return (x, iterations, converged_bool)
    """
    n = len(b)
    x = x0.copy().astype(float)

    for k in range(max_iterations):
        x_new = x.copy()

        for i in range(n):
            # s1 uses already-updated values (0..i-1)
            s1 = sum(A[i][j] * x_new[j] for j in range(i))
            # s2 uses old values (i+1..n-1)
            s2 = sum(A[i][j] * x[j] for j in range(i + 1, n))

            x_new[i] = (b[i] - s1 - s2) / A[i][i]

        # convergence check (looked up)
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new, (k + 1), True

        # blow-up check (looked this up ig safety check)
        if (not np.all(np.isfinite(x_new))) or (np.linalg.norm(x_new, ord=np.inf) > blowup):
            return x_new, (k + 1), False

        x = x_new

    return x, max_iterations, False


def run(case_name, A, b, tol=1e-6):
    print(f"{case_name} ")
    x0 = np.zeros_like(b, dtype=float)  # starting guess = all zeros 

    # Jacobi timing
    t0 = time.perf_counter()
    x_j, it_j, ok_j = jacobi(A, b, x0, tol=tol)
    t_j = time.perf_counter() - t0

    # Gauss-Seidel timing
    t0 = time.perf_counter()
    x_gs, it_gs, ok_gs = gauss_seidel(A, b, x0, tol=tol)
    t_gs = time.perf_counter() - t0

    print("Jacobi:")
    print("  converged", ok_j)
    print("  iterations:", it_j)
    print("  time (sec):", t_j)
    print("  x:", x_j)

    print("Gauss-Seidel:")
    print("  converged", ok_gs)
    print("  iterations:", it_gs)
    print("  time (sec):", t_gs)
    print("  x:", x_gs)



# Challenge 6a
A6a = np.array([
    [10, -1,  2,  0,  0],
    [-1, 11, -1,  3,  0],
    [ 2, -1, 10, -1,  0],
    [ 0,  3, -1,  8, -2],
    [ 0,  0,  0, -2,  9]
], dtype=float)

b6a = np.array([14, 30, 26, 25, 37], dtype=float)



# Challenge 6b 

A6b = np.array([
    [1, 2, 3, 0, 0],
    [2, 1, 2, 3, 0],
    [3, 2, 1, 2, 3],
    [0, 3, 2, 1, 2],
    [0, 0, 3, 2, 1]
], dtype=float)

b6b = np.array([14, 22, 33, 26, 22], dtype=float)


run("Challenge 6a", A6a, b6a, tol=1e-6)
run("Challenge 6b", A6b, b6b, tol=1e-6)

Challenge 6a 
Jacobi:
  converged True
  iterations: 19
  time (sec): 0.0017307959496974945
  x: [0.99999995 2.00000025 2.99999986 4.00000008 4.99999983]
Gauss-Seidel:
  converged True
  iterations: 11
  time (sec): 0.0011987928301095963
  x: [1.00000003 2.00000002 2.99999999 3.99999998 5.        ]
Challenge 6b 
Jacobi:
  converged False
  iterations: 14
  time (sec): 0.0008197501301765442
  x: [-2.02235411e+12 -2.50494612e+12 -3.11754299e+12 -2.50494612e+12
 -2.02235411e+12]
Gauss-Seidel:
  converged False
  iterations: 11
  time (sec): 0.0011672340333461761
  x: [ 9.79473379e+11 -3.22543723e+12  3.42397016e+12  3.58168948e+12
 -1.74352895e+13]
